# Lab 2 — Trace + test with diverse inputs (EP06)

Two design principles from §19.3:
- **② Transparency over cleverness** — every step of an agent must be inspectable → keep a **trace** of what each step thought, which tool it called and what came back
- **⑥ Test with diverse inputs** — agent behaviour varies → test with edge cases, ambiguous requests and malformed inputs, and run each several times

**What you'll learn**
1. Add a trace to the agent loop and print it step by step
2. Use the trace to find a bug where the answer looks right but is wrong
3. A 7-case × 3-round test suite that checks its own results
4. Use the trace of a failing case to find where it broke, fix it, and rerun the suite

The paper library is **simulated data written for teaching** (the same one as Lab 1).
Every tool call and model answer comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.3 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **6 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep06-lab2-trace-and-test.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs well under one cent) |
| Packages | `openai`, `python-dotenv` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

How to set up the key, and why we chose this model, is in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

## 1. From Lab 1

The same model + provider (Gemma 4 26B pinned to DeepInfra) · the 12-paper library · `find_papers` · the clearly written `search_papers` tool

In [ ]:
import json, os, time, datetime
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}}

In [ ]:
PAPERS = [
    {"id": "P01", "title": "Informer: efficient transformer for long sequence time-series forecasting", "year": 2021},
    {"id": "P02", "title": "Autoformer: decomposition transformers for long-term series forecasting", "year": 2021},
    {"id": "P03", "title": "FEDformer: frequency enhanced decomposed transformer for time series", "year": 2022},
    {"id": "P04", "title": "Are transformers effective for time series forecasting?", "year": 2023},
    {"id": "P05", "title": "PatchTST: a time series is worth 64 words", "year": 2023},
    {"id": "P06", "title": "iTransformer: inverted transformers for time series forecasting", "year": 2024},
    {"id": "P07", "title": "TimesFM: a decoder-only foundation model for time-series forecasting", "year": 2024},
    {"id": "P08", "title": "Chronos: learning the language of time series", "year": 2024},
    {"id": "P09", "title": "Attention is all you need", "year": 2017},
    {"id": "P10", "title": "BERT: pre-training of deep bidirectional transformers", "year": 2019},
    {"id": "P11", "title": "Temporal fusion transformers for interpretable time series forecasting", "year": 2019},
    {"id": "P12", "title": "Moirai: unified training of universal time series forecasting transformers", "year": 2024},
]
STOP = {"with", "from", "paper", "papers", "model", "models", "using", "published", "since", "research", "about"}

def find_papers(query, year_from=0, library=PAPERS):
    words = [w for w in query.lower().replace("-", " ").split() if len(w) > 3 and w not in STOP]
    return [p for p in library
            if p["year"] >= year_from and any(w in p["title"].lower().replace("-", " ") for w in words)]

TOOLS = [{"type": "function", "function": {
    "name": "search_papers",
    "description": "Search academic papers in the Cogito Learn library by topic. Returns a list of {id, title, year}; returns [] if none match.",
    "parameters": {"type": "object", "properties": {
        "query": {"type": "string", "description": "Topic in English, e.g. 'time series transformer'"},
        "year_from": {"type": "integer", "description": "Earliest publication year wanted (CE), e.g. 2023 = from 2023 onward"}},
    "required": ["query", "year_from"]}}}]

def run_tool(name, args):
    return find_papers(args["query"], args["year_from"])

SYSTEM = "You are the Cogito Learn paper-search assistant. Always use tools to get real data; never answer from memory."

## 2. Adding a trace to the agent loop

The same loop as Lab 1, plus a `trace` list that records every step:
- an **LLM** step — how many seconds and tokens it took, and which tool the model asked for, or what it answered
- a **tool** step — which tool was called, with which arguments, and what came back

In [ ]:
def run_agent(question, max_steps=6):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    trace = []
    for step in range(1, max_steps + 1):
        t0 = time.time()
        reply = client.chat.completions.create(model=MODEL, extra_body=EXTRA, messages=messages, tools=TOOLS)
        msg = reply.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        # ✏️ TODO 1 (▶ video 1:49): Record the LLM step in the trace: how many seconds it took and which tools the model asked for
        trace.append({"step": step, "kind": "llm", "sec": ___,
                      "tokens": reply.usage.total_tokens,
                      "tools": ___, "text": msg.content})
        if not msg.tool_calls:
            return msg.content, trace
        # ✏️ TODO 2 (▶ video 2:09): Record the tool step in the trace: the tool's name, its arguments and its result
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = run_tool(call.function.name, args)
            trace.append({"step": step, "kind": "tool", "name": ___, "args": ___, "result": ___})
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result, ensure_ascii=False)})
    return "stopped: step limit reached", trace

In [ ]:
def show_trace(trace):
    for t in trace:
        if t["kind"] == "llm":
            what = f"asks for tool {t['tools']}" if t["tools"] else f"answers: {t['text'][:70]!r}"
            print(f"step {t['step']} 🧠 LLM ({t['sec']} s, {t['tokens']} tokens) → {what}")
        else:
            ids = [p["id"] for p in t["result"]]
            print(f"step {t['step']} 🔧 {t['name']}({t['args']}) → {len(ids)} papers {ids}")

## 3. An answer that looks right, but the trace says otherwise

Ask for "the last 2 years" — the answer looks fine, but which year_from did the model send?

In [ ]:
RECENT = "Find papers on transformers for time series from the last 2 years"
answer, trace = run_agent(RECENT)
print(answer)

In [ ]:
show_trace(trace)

In [ ]:
# ✏️ TODO 3 (▶ video 3:23): Tell the model today's date by appending it to SYSTEM
SYSTEM += ___

answer, trace = run_agent(RECENT)
show_trace(trace)

## 4. A test suite: 7 inputs × 3 rounds

| Case | Input | Passes when |
|---|---|---|
| normal | from 2023 on | PatchTST is in, Informer (2021) is not |
| last 2 years | the last 2 years | year_from ≥ this year − 2 |
| ambiguous | "find me some good papers" | no blind search (it asks back) |
| empty | an empty message | no tool call |
| year 2099 | a year that hasn't come yet | no paper named in the answer |
| year range | 2021 to 2022 | Autoformer is in, PatchTST (2023) is not |
| not in library | reinforcement learning | no paper named in the answer (the library has none on it) |

In [ ]:
NAMES = ["Informer", "Autoformer", "FEDformer", "PatchTST", "iTransformer", "TimesFM", "Chronos", "Moirai"]
THIS_YEAR = datetime.date.today().year

def named(answer):
    return [n for n in NAMES if n in answer]

def searched(trace):
    return [t for t in trace if t["kind"] == "tool"]

In [ ]:
CASES = [
    ("normal",       "Find papers on transformers for time series published in 2023 or later",
                   lambda a, tr: "PatchTST" in a and "Informer" not in a),
    ("last 2 years", RECENT,
                   lambda a, tr: all(t["args"]["year_from"] >= THIS_YEAR - 2 for t in searched(tr))),
    ("ambiguous",      "Find me some good papers",               lambda a, tr: not searched(tr)),
    ("empty",       "",                                   lambda a, tr: not searched(tr)),
    ("year 2099",    "Give me time-series papers from 2099", lambda a, tr: not named(a)),
    ("year range",     "Find time-series papers published between 2021 and 2022",
                   lambda a, tr: "Autoformer" in a and "PatchTST" not in a),
    # ✏️ TODO 4 (▶ video 5:09): Write the check for the not-in-library case: the answer must not name any library paper
    ("not in library",    "Find me papers on reinforcement learning", lambda a, tr: ___),
]

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def check(case):
    name, question, ok = case
    answer, trace = run_agent(question)
    return name, ok(answer, trace), answer, trace

# ✏️ TODO 5 (▶ video 5:30): Build the job list so each case appears rounds times

def run_suite(rounds=3):
    jobs = ___
    with ThreadPoolExecutor(8) as pool:
        results = list(pool.map(check, jobs))
    for name, _, _ in CASES:
        marks = "".join("✅" if ok else "❌" for n, ok, a, tr in results if n == name)
        print(f"{name:15} {marks}")
    return results

In [ ]:
results = run_suite()

## 5. Using the trace to find where it broke

In [ ]:
for name, ok, answer, trace in results:
    if not ok:
        show_trace(trace)
        print(answer[:200])
        break

In [ ]:
print(find_papers("reinforcement learning"))

In [ ]:
# ✏️ TODO 6 (▶ video 7:00): Add the over-broad word to the STOP set
___
print(find_papers("reinforcement learning"))

In [ ]:
results = run_suite()

## Summary

| Principle | What we saw in the lab |
|---|---|
| ② trace | "the last 2 years" → the answer looked fine, but the trace showed year_from was too old, because the model didn't know the date → put the date in the system prompt |
| ⑥ diverse inputs | 7 cases × 3 rounds · Gemma handled the ambiguous request, the empty message, 2099 and the year range by itself · the not-in-library case failed every round |
| ② + ⑥ | the failing case's trace pointed to `find_papers` (the word "learning"), not the model → fix it and rerun the whole suite: 21/21 passed |

**Exercises**
1. Add 2 cases of your own (e.g. a Buddhist-era year, or "the 3 latest papers") with check functions, and run the suite
2. Instead of adding `learning` to `STOP`, make `find_papers` require **at least 2 matching words**, then run the suite — does every case pass? If not, open the trace and find out why

**Next — Lab 3:** use Table 19.3 to choose a pattern for a real task, measuring LLM calls and cost